# Training habits: batches, Adam and early stopping
**Machine Learning · Week 5 (Neural networks with PyTorch) · Day 4**

Lesson page: https://sara-aminpour.com/tutorials/ml/w5d4.html

Three habits that make training work in practice: **mini-batches**, a smarter optimizer
(**momentum** and **Adam**), and **early stopping**. All three are a line or two of PyTorch.

In [ ]:
import copy
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from torch.utils.data import TensorDataset, DataLoader

torch.set_num_threads(1)
X, y = make_moons(n_samples=2000, noise=0.25, random_state=0)
data = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32).reshape(-1, 1))
loader = DataLoader(data, batch_size=32, shuffle=True)   # shuffle: a fresh random order every epoch

xb, yb = next(iter(loader))
print("one batch:", tuple(xb.shape), "inputs and", tuple(yb.shape), "labels")
print("steps in one epoch:", len(loader), "(2000 examples in batches of 32, the last batch is smaller)")

## Same number of epochs, many more steps
One epoch is one pass through the data. With the whole dataset per step, that's a single step.
With batches of 32, it's 63 cheaper, noisier steps.

In [ ]:
def make_net(seed=0):
    torch.manual_seed(seed)
    return torch.nn.Sequential(torch.nn.Linear(2, 32), torch.nn.Tanh(), torch.nn.Linear(32, 1))

loss_fn = torch.nn.BCEWithLogitsLoss()
X_all, y_all = data.tensors

def full_loss(net):
    with torch.no_grad():
        return loss_fn(net(X_all), y_all).item()

net = make_net()
opt = torch.optim.SGD(net.parameters(), lr=0.5)
for epoch in range(10):
    opt.zero_grad()
    loss_fn(net(X_all), y_all).backward()          # the whole dataset: one step per epoch
    opt.step()
print(f"whole dataset per step, 10 epochs = 10 steps:  loss {full_loss(net):.3f}")

net = make_net()
opt = torch.optim.SGD(net.parameters(), lr=0.5)
for epoch in range(10):
    for xb, yb in loader:                          # batches of 32: 63 steps per epoch
        opt.zero_grad()
        loss_fn(net(xb), yb).backward()
        opt.step()
print(f"batches of 32,          10 epochs = {10 * len(loader)} steps: loss {full_loss(net):.3f}")

## Three optimizers, one network
Same starting weights, same batches. Only the update rule changes.

In [ ]:
def train(make_opt, epochs=15):
    net = make_net()
    opt = make_opt(net.parameters())
    order = torch.Generator().manual_seed(1)       # the same batches for every optimizer
    curve = [full_loss(net)]
    for epoch in range(epochs):
        for xb, yb in DataLoader(data, batch_size=32, shuffle=True, generator=order):
            opt.zero_grad()
            loss_fn(net(xb), yb).backward()
            opt.step()
        curve.append(full_loss(net))
    return curve

curves = {
    "SGD (lr 0.05)": train(lambda p: torch.optim.SGD(p, lr=0.05)),
    "SGD + momentum 0.9 (lr 0.05)": train(lambda p: torch.optim.SGD(p, lr=0.05, momentum=0.9)),
    "Adam (lr 0.01)": train(lambda p: torch.optim.Adam(p, lr=0.01)),
}
fig, ax = plt.subplots(figsize=(6.4, 3.4), layout="constrained")
for (name, curve), color in zip(curves.items(), ["#ff8a50", "#7b5cfa", "#17a98c"]):
    ax.plot(curve, label=name, color=color, lw=2.5)
    print(f"{name:30s} loss after 15 epochs: {curve[-1]:.3f}")
ax.set_xlabel("epoch")
ax.set_ylabel("loss on all 2000 points")
ax.legend()
plt.show()

## A recipe for overfitting
40 noisy training points and a network with 4,417 weights. We watch the loss on the training
points and on 300 fresh **validation** points after every epoch.

In [ ]:
T = lambda a: torch.tensor(a, dtype=torch.float32)
Xtr, ytr = make_moons(40, noise=0.35, random_state=2)
Xva, yva = make_moons(300, noise=0.35, random_state=102)
Xte, yte = make_moons(2000, noise=0.35, random_state=202)
Xtr, Xva, Xte = T(Xtr), T(Xva), T(Xte)
ytr, yva, yte = T(ytr).reshape(-1, 1), T(yva).reshape(-1, 1), T(yte).reshape(-1, 1)

def big_net():
    torch.manual_seed(0)
    return torch.nn.Sequential(torch.nn.Linear(2, 64), torch.nn.Tanh(), torch.nn.Linear(64, 64), torch.nn.Tanh(), torch.nn.Linear(64, 1))

def one_epoch(net, opt, order):
    perm = torch.randperm(len(Xtr), generator=order)
    for i in range(0, len(Xtr), 8):                # batches of 8
        idx = perm[i:i + 8]
        opt.zero_grad()
        loss_fn(net(Xtr[idx]), ytr[idx]).backward()
        opt.step()

def evaluate(net, X, y):
    with torch.no_grad():
        out = net(X)
        return loss_fn(out, y).item(), ((out > 0).float() == y).float().mean().item()

net = big_net()
opt = torch.optim.Adam(net.parameters(), lr=0.002)
order = torch.Generator().manual_seed(0)
train_curve, val_curve = [], []
for epoch in range(400):
    train_curve.append(evaluate(net, Xtr, ytr)[0])
    val_curve.append(evaluate(net, Xva, yva)[0])
    one_epoch(net, opt, order)

best = int(np.argmin(val_curve))
fig, ax = plt.subplots(figsize=(6.4, 3.4), layout="constrained")
ax.plot(train_curve, color="#17a98c", lw=2.5, label="training loss")
ax.plot(val_curve, color="#7b5cfa", lw=2.5, label="validation loss")
ax.axvline(best, color="#2c2842", ls="--", lw=1.2)
ax.annotate(f"lowest validation loss: epoch {best}", (best, val_curve[best]), xytext=(best + 20, 0.63),
            arrowprops=dict(arrowstyle="->", color="#2c2842", lw=1.2))
ax.set_xlabel("epochs trained")
ax.set_ylabel("loss")
ax.legend()
plt.show()
print(f"lowest validation loss after {best} epochs; test accuracy after all 400 epochs: {evaluate(net, Xte, yte)[1]:.3f}")

## Your turn: early stopping
Train the same network again, but after each epoch check the validation loss. Keep a copy of the
best weights so far, stop once there has been no improvement for `patience` epochs, and put the
best weights back at the end.

In [ ]:
net = big_net()
opt = torch.optim.Adam(net.parameters(), lr=0.002)
order = torch.Generator().manual_seed(0)
patience = 20
best_loss, best_epoch, best_state, waited = float("inf"), 0, None, 0
for epoch in range(400):
    val_loss = evaluate(net, Xva, yva)[0]
    # TODO: if val_loss < best_loss, remember it, the epoch and a copy of the weights
    #       (copy.deepcopy(net.state_dict())), and set waited back to 0. Otherwise add 1 to waited.
    # TODO: break out of the loop once waited reaches patience
    one_epoch(net, opt, order)
# TODO: put the best weights back: net.load_state_dict(best_state)
print(f"stopped at epoch {epoch}, kept the weights from epoch {best_epoch}")
print(f"test accuracy: {evaluate(net, Xte, yte)[1]:.3f}")

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
net = big_net()
opt = torch.optim.Adam(net.parameters(), lr=0.002)
order = torch.Generator().manual_seed(0)
patience = 20
best_loss, best_epoch, best_state, waited = float("inf"), 0, None, 0
for epoch in range(400):
    val_loss = evaluate(net, Xva, yva)[0]
    if val_loss < best_loss:
        best_loss, best_epoch, waited = val_loss, epoch, 0
        best_state = copy.deepcopy(net.state_dict())   # a real copy, not a reference that keeps changing
    else:
        waited += 1
        if waited >= patience:
            break
    one_epoch(net, opt, order)
net.load_state_dict(best_state)
print(f"stopped at epoch {epoch}, kept the weights from epoch {best_epoch} (validation loss {best_loss:.3f})")
print(f"test accuracy with early stopping: {evaluate(net, Xte, yte)[1]:.3f}")
print("training all 400 epochs gave 0.725: stopping early kept the boundary smooth, and it took a quarter of the time.")